In [11]:
from pytrends.request import TrendReq
import pandas as pd

print("Fetching Google Trends data for Asthma (Delhi)...")

# Initialize TrendReq with built-in retry logic
# retries=3: It will attempt the request 3 additional times if it hits a 429 error.
# backoff_factor=2: It will exponentially increase the wait time between retries (e.g., 2s, 4s, 8s).
pytrends = TrendReq(hl='en-IN', tz=330, retries=3, backoff_factor=2)

# Set the keyword and location payload
pytrends.build_payload(["asthma"], timeframe='today 3-m', geo='IN-DL')

# Fetch the temporal data
trends_df = pytrends.interest_over_time()

# Clean up the dataframe
if 'isPartial' in trends_df.columns:
    trends_df = trends_df.drop(columns=['isPartial'])

# Display the clean table
trends_df.tail()


Fetching Google Trends data for Asthma (Delhi)...


,asthma
date,
2026-05-22,41
2026-05-23,30
2026-05-24,36
2026-05-25,33
2026-05-26,30


In [12]:
import requests
import pandas as pd
from datetime import datetime, timedelta

print("Fetching historical PM2.5 and AQI data for Delhi...")

# 1. Dynamically calculate today and 90 days ago
end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')

lat, lon = 28.6139, 77.2090

# 2. Inject start_date and end_date into the URL string
url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&hourly=pm2_5,us_aqi&timezone=Asia%2FCalcutta&start_date={start_date}&end_date={end_date}"

response = requests.get(url)
data = response.json()

aqi_df = pd.DataFrame(data['hourly'])

# 3. Clean the data! .dropna() permanently removes any rows that contain NaN
aqi_df = aqi_df.dropna()

print(f"Data alignment complete! Fetched {len(aqi_df)} hours of historical data.")
display(aqi_df.tail(10))

Fetching historical PM2.5 and AQI data for Delhi...
Data alignment complete! Fetched 2184 hours of historical data.


,time,pm2_5,us_aqi
2174,2026-05-27T14:00,93.4,427
2175,2026-05-27T15:00,114.3,442
2176,2026-05-27T16:00,136.4,464
2177,2026-05-27T17:00,159.3,487
2178,2026-05-27T18:00,182.5,513
2179,2026-05-27T19:00,202.1,539
2180,2026-05-27T20:00,217.5,565
2181,2026-05-27T21:00,224.9,589
2182,2026-05-27T22:00,226.8,610
2183,2026-05-27T23:00,225.8,625


In [ ]:
 # Cell 1: Unified Data Fetching Pipeline
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from pytrends.request import TrendReq

print("--- INITIALIZING BIOTECH DATA PIPELINE ---")

# 1. SET THE TIME WINDOW (Last 90 Days)
end_date = datetime.today().strftime('%Y-%m-%d')
start_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')
print(f"Time Window: {start_date} to {end_date}")

# 2. FETCH GOOGLE TRENDS DATA (Temporal RSV)
print("\nFetching Google Trends RSV for 'Asthma' in Delhi...")
pytrends = TrendReq(hl='en-IN', tz=330)
try:
    pytrends.build_payload(["asthma"], timeframe='today 3-m', geo='IN-DL')
    trends_df = pytrends.interest_over_time()
    if 'isPartial' in trends_df.columns:
        trends_df = trends_df.drop(columns=['isPartial'])
    print(f"Successfully fetched {len(trends_df)} days of RSV data.")
except Exception as e:
    print("Google API Rate Limited. Please run backup cache.")

# 3. FETCH OPEN-METEO ENVIRONMENTAL DATA (AQI & PM2.5)
print("\nFetching Environmental Baseline from Open-Meteo...")
lat, lon = 28.6139, 77.2090
url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&hourly=pm2_5,us_aqi&timezone=Asia%2FCalcutta&start_date={start_date}&end_date={end_date}"

response = requests.get(url)
data = response.json()
aqi_df = pd.DataFrame(data['hourly']).dropna()
print(f"Successfully fetched {len(aqi_df)} hours of Air Quality data.")

print("\n--- PIPELINE EXECUTION COMPLETE ---")